In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dhruvildave/en-fr-translation-dataset")

print("Path to dataset files:", path)

KeyboardInterrupt: 

In [1]:
import os

data_path = "/root/.cache/kagglehub/datasets/dhruvildave/en-fr-translation-dataset/versions/2"
os.listdir(data_path)


['en-fr.csv']

In [6]:
import pandas as pd
import os

file_path = os.path.join(data_path, "en-fr.csv")

chunksize = 100000  # number of rows per chunk
source_sentences = []
target_sentences = []

max_samples = 40000  # only get 40k rows

for chunk in pd.read_csv(file_path, chunksize=chunksize):
    source_sentences.extend(chunk['en'].tolist())
    target_sentences.extend(chunk['fr'].tolist())

    # Stop when we reach max_samples
    if len(source_sentences) >= max_samples:
        source_sentences = source_sentences[:max_samples]
        target_sentences = target_sentences[:max_samples]
        break

print(len(source_sentences), len(target_sentences))


40000 40000


In [9]:
import re
import string

def clean_text(text):
    text = str(text)  # convert float/NaN to string
    text = text.lower().strip()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


source_sentences = [clean_text(s) for s in source_sentences]
target_sentences = [clean_text(t) for t in target_sentences]


In [10]:
target_sentences = ['<sos> ' + t + ' <eos>' for t in target_sentences]


In [11]:
target_sentences

['<sos> il a transformé notre vie il a transformé la société son fonctionnement la technologie moteur du changement accueil concepts enseignants recherche aperçu collaborateurs web hhcc ressources commentaires musée virtuel du canada <eos>',
 '<sos> plan du site <eos>',
 '<sos> rétroaction <eos>',
 '<sos> crédits <eos>',
 '<sos> english <eos>',
 '<sos> qu’estce que la lumière <eos>',
 '<sos> la découverte du spectre de la lumière blanche des codes dans la lumière le spectre électromagnétique les spectres d’émission les spectres d’absorption les annéeslumière la pollution lumineuse <eos>',
 '<sos> le ciel des premiers habitants la vision contemporaine de lunivers l’astronomie pour tous <eos>',
 '<sos> bande dessinée <eos>',
 '<sos> liens <eos>',
 '<sos> glossaire <eos>',
 '<sos> observatoires <eos>',
 '<sos> astronomes introduction vidéo dintroduction questce que lastronomie <eos>',
 '<sos> souvent considérée comme la plus ancienne des sciences elle découle de notre étonnement et de nos

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# English tokenizer
src_tokenizer = Tokenizer()
src_tokenizer.fit_on_texts(source_sentences)
source_sequences = src_tokenizer.texts_to_sequences(source_sentences)
source_vocab_size = len(src_tokenizer.word_index) + 1

# French tokenizer
tgt_tokenizer = Tokenizer()
tgt_tokenizer.fit_on_texts(target_sentences)
target_sequences = tgt_tokenizer.texts_to_sequences(target_sentences)
target_vocab_size = len(tgt_tokenizer.word_index) + 1


In [13]:
max_src_len = max(len(seq) for seq in source_sequences)
max_tgt_len = max(len(seq) for seq in target_sequences)

from tensorflow.keras.preprocessing.sequence import pad_sequences

source_sequences = pad_sequences(source_sequences, maxlen=max_src_len, padding='post')
target_sequences = pad_sequences(target_sequences, maxlen=max_tgt_len, padding='post')


In [14]:
decoder_input = target_sequences[:, :-1]
decoder_output = target_sequences[:, 1:]


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

# =====================
# PARAMETERS
# =====================
embedding_dim = 256
latent_dim = 512
batch_size = 64
epochs = 20
max_src_len = source_sequences.shape[1]
max_tgt_len = decoder_input.shape[1]
source_vocab_size = len(src_tokenizer.word_index) + 1
target_vocab_size = len(tgt_tokenizer.word_index) + 1

# =====================
# ENCODER
# =====================
encoder_inputs = Input(shape=(max_src_len,))
enc_emb = Embedding(input_dim=source_vocab_size, output_dim=embedding_dim, mask_zero=True)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# =====================
# DECODER
# =====================
decoder_inputs = Input(shape=(max_tgt_len,))
dec_emb = Embedding(input_dim=target_vocab_size, output_dim=embedding_dim, mask_zero=True)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(target_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# =====================
# MODEL
# =====================
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# =====================
# TRAINING
# =====================
model.fit(
    [source_sequences, decoder_input],
    decoder_output[..., None],  # Add last dimension for sparse_categorical_crossentropy
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)
